In [ ]:
# ========================================================
# 1. INSTALL LIBRARY & OLLAMA
# ========================================================
!apt-get update && apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain langchain_community langchain-ollama langchainhub
!pip install sentence-transformers langchain-huggingface bert-score

In [ ]:
import os
import subprocess
import time
import json
import torch
import numpy as np
from google.colab import drive, userdata
from operator import itemgetter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings # Pengganti Google Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama
from bert_score import score

In [ ]:
# Jalankan server Ollama di background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
!ollama pull qwen2.5:3b

In [ ]:
# ========================================================
# 2. LOAD DATA & VECTOR STORE
# ========================================================
# Mount drive dan load data (Sesuaikan path file Anda)
drive.mount('/content/drive')
with open('/content/drive/MyDrive/Projek Klasifikasi Gambar/faq_pajak_final_v2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
# Inisialisasi Embedding BGE-M3 (Sangat akurat untuk Bahasa Indonesia)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
# Buat dokumen dan simpan ke Vector Store
documents = [Document(page_content=f"Question: {item['question']}\nAnswer: {item['answer']}") for item in data]
vectorstore = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Mengambil 3 dokumen teratas

In [ ]:
# ========================================================
# 3. DEFINISI MODEL & ALUR RAG (HyDE)
# ========================================================
llm = ChatOllama(model="qwen2.5:3b", temperature=0)

In [ ]:
# Langkah 1: HyDE (Membuat teks bantuan pencarian)
hyde_template = "Tulis sebuah paragraf makalah ilmiah untuk menjawab pertanyaan tersebut: {question}"
prompt_hyde = ChatPromptTemplate.from_template(hyde_template)
generate_hyde = (prompt_hyde | llm | StrOutputParser())

In [ ]:
# Langkah 2: Final Answer (Menggunakan konteks asli)
final_template = """Gunakan konteks berikut untuk menjawab pertanyaan.
Jika tidak ada di konteks, jawablah 'Pertanyaan Anda diteruskan ke Human Assistant untuk bantuan lebih lanjut.

Konteks:
{context}

Pertanyaan: {question}
Jawaban:"""
prompt_final = ChatPromptTemplate.from_template(final_template)

In [ ]:
# Chain Utama
def run_full_rag(question):
    # A. Jalankan HyDE
    hypothetical_doc = generate_hyde.invoke({"question": question})

    # B. Cari dokumen berdasarkan teks HyDE
    docs = retriever.invoke(hypothetical_doc)
    context_text = "\n\n".join([d.page_content for d in docs])

    # C. Generate jawaban akhir
    final_answer = (prompt_final | llm | StrOutputParser()).invoke({
        "context": context_text,
        "question": question
    })

    return {
        "hyde": hypothetical_doc,
        "docs": docs,
        "answer": final_answer
    }

In [ ]:
# ========================================================
# 4. UJI COBA OUTPUT MODEL
# ========================================================
pertanyaan = "jelaskan secara singkat apa saja isi faq ini" # Ganti dengan pertanyaan Anda
result = run_full_rag(pertanyaan)

print("\n" + "="*50)
print(f"PERTANYAAN: {pertanyaan}")
print("="*50)

print(f"\n[STEP 1: HyDE - Apa yang dipikirkan model?]\n{result['hyde']}")

print(f"\n[STEP 2: RETRIEVAL - Dokumen yang ditemukan dari dataset]")
for i, d in enumerate(result['docs']):
    print(f"{i+1}. {d.page_content[:150]}...")

print(f"\n[STEP 3: JAWABAN AKHIR]\n{result['answer']}")

In [ ]:
# ========================================================
# 5. EVALUASI BERT SCORE (Contoh 1 data)
# ========================================================
ref_answer = data[0]['answer'] # Ambil jawaban asli dari dataset sebagai referensi
P, R, F1 = score([result['answer']], [ref_answer], lang="id", verbose=False)

In [ ]:
# Tampilkan Hasil Akhir
print("\n" + "="*40)
print(f"MODEL: Qwen2.5:3B | EMBEDDING: BGE-M3")
print("="*40)
print(f"Rata-rata Precision: {P.mean().item():.4f}")
print(f"Rata-rata Recall:    {R.mean().item():.4f}")
print(f"Rata-rata F1 Score:  {F1.mean().item():.4f}")
print("="*40)